# 🍜 頑固ラーメン屋のAIを作ろう — 最終章

---
# 📕 ラボ3：実用SLMへ — LoRAファインチューニング & MCP入門
### (Lab 3: A Practical SLM — LoRA Fine-Tuning & Intro to MCP)
---

lab2で作ったモデルは「ゼロから」学習したため、あなたのFAQの文字パターンしか知りません。
日本語そのものは知らないのです。

このラボでは発想を変えます：

> **すでに日本語を知っている「事前学習済みSLM」を借りてきて、
> あなたのFAQデータで「微調整（ファインチューニング）」する。**

これが、実務で最も使われるアプローチです。

## 📖 このラボの辞書 (Concept Dictionary)

| 言葉 (Term) | 意味 (Meaning) | 例え (Analogy) |
| :--- | :--- | :--- |
| **事前学習済みモデル (Pre-trained Model)** | 大量のテキストで訓練済みの汎用モデル | 日本語ペラペラの新入社員 |
| **ファインチューニング (Fine-tuning)** | 自分のデータで追加訓練すること | 新入社員への自社研修 |
| **LoRA (ローラ)** | 少ないメモリで微調整する省エネ技術 | 教科書を書き換えず、**付箋を貼る** |
| **Hugging Face** | 学習済みモデルの共有サイト | AIモデルのGitHub |
| **MCP** | AIと外部ツールを繋ぐ標準規格 | AI用のUSB規格 |

## 🔑 LoRA (Low-Rank Adaptation) とは？

0.5Bモデルでも約5億個のパラメータがあります。全部を再訓練するのは大変です。

LoRAは、**元のモデルは凍結（変更しない）** したまま、各層に **小さな「差分アダプタ」** を
追加して、そこだけを訓練します。

```text
 通常のファインチューニング:  5億個のパラメータ全部を書き換える 😰
 LoRA                      :  元のモデルは凍結 + 数百万個の「付箋」だけ訓練 😊
                              → メモリ小・高速・元のモデルを壊さない
```

保存されるのは付箋（アダプタ）だけなので、ファイルサイズも数MB〜数十MBと軽量です。


# ==============================================================================
# ✅ ステップ0：ライブラリのインストールと読み込み
# ==============================================================================
本ラボでは Hugging Face のライブラリを使います：

- **transformers**: 学習済みモデルの読み込み・実行
- **peft** (Parameter-Efficient Fine-Tuning): LoRAの実装
- **accelerate**: GPU利用の自動最適化


In [ ]:
# 初回のみ実行してください（1〜2分かかります）
%pip install -q -U transformers peft accelerate

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"使用デバイス: {device}")
if device == 'cpu':
    print("⚠️ GPUが見つかりません。CPUでも動きますが、訓練に時間がかかります。")

# ==============================================================================
# ✅ ステップ1：事前学習済みSLMをダウンロードする
# ==============================================================================

環境に合わせて、2つのSLMから **自動で** 選択します（次のセルが判定します）。

| | GPU がある場合 | CPU のみの場合 |
| :--- | :--- | :--- |
| モデル | **Qwen2.5-0.5B-Instruct** | **llm-jp-3-150m-instruct3** |
| 開発元 | Alibaba | 国立情報学研究所 (LLM-jp) 🇯🇵 |
| パラメータ数 | 5億 | 1.5億 |
| ダウンロード | 約1GB | **約300MB** |
| 特徴 | 多言語で高性能 | 日本語特化・軽量でCPUでも訓練可能 |

どちらも「Instruct」= 指示に従って会話するよう調整済みのモデルです。
（今朝のBigramは数万、lab2のTransformerは約20万パラメータでした。桁が違います！）

> 💡 初回はモデルのダウンロードに数分かかります。
> **⚠️ もしダウンロード中や読み込み中にカーネルが固まったら**: インスタンスのメモリ不足が
> 原因です。Kernel → Restart Kernel で再起動し、下の `FORCE_SMALL_MODEL = True` にして
> 再実行してください（ダウンロード済みの部分はキャッシュから再開されます）。



In [ ]:
# 環境に合わせてモデルを自動選択します
# メモリの少ないインスタンスで大きいモデルを読むと固まるため、CPU環境では軽量モデルを使います
FORCE_SMALL_MODEL = False   # ← 固まった経験がある場合は True にしてください

if device == 'cuda' and not FORCE_SMALL_MODEL:
    model_name = "Qwen/Qwen2.5-0.5B-Instruct"        # GPU: 5億パラメータ
else:
    model_name = "llm-jp/llm-jp-3-150m-instruct3"    # CPU: 日本語特化・1.5億パラメータ

print(f"使用モデル: {model_name}")

# トークナイザ: 文字→トークンの変換器。
# 今朝は自分で作りましたが（stoi/itos）、実用モデルには専用の高性能版が付属します。
# 文字単位ではなく「サブワード単位」（例:「営業時間」→ 1〜2トークン）で効率的に扱います
tokenizer = AutoTokenizer.from_pretrained(model_name)

# モデル本体
base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype=torch.float32,   # 数値精度。まずは標準のfloat32で安全に
).to(device)

n_params = sum(p.numel() for p in base_model.parameters())
print(f"モデル読み込み完了！ パラメータ数: {n_params:,} 個")
print(f"（lab2のモデルの約 {n_params // 200_000:,} 倍の規模です）")

# トークナイザの動きを覗いてみましょう
sample = "営業時間を教えてください"
ids = tokenizer(sample)["input_ids"]
print(f"\n『{sample}』(12文字) → {len(ids)} トークン: {ids}")
print("→ 今朝の文字単位より、ずっと効率的に日本語を扱っています")

# ==============================================================================
# ✅ ステップ2：訓練前のモデルに質問してみる (Before)
# ==============================================================================

まず、**訓練前** のモデルにあなたの会社のことを質問してみましょう。
一般常識は答えられますが、あなたの会社の固有情報は知らないはず（または適当な作り話をする＝
**ハルシネーション**）です。この「Before」を覚えておいてください。

> 💡 CPU版の軽量モデル（1.5億パラメータ）は、一般会話もかなり素朴な回答しかできません。
> それでも「訓練後にあなたのFAQを答えられるようになる」対比は十分に体験できます。


In [ ]:
def ask_model(model, question, max_new_tokens=100):
    """Instructモデルへの質問関数。チャットテンプレートという正式な書式で問いかけます"""
    messages = [
        {"role": "system", "content": "あなたは親切なFAQアシスタントです。簡潔に日本語で答えてください。"},
        {"role": "user", "content": question},
    ]
    # チャットテンプレート: 会話を、モデルが訓練された正式な形式のテキストに変換する仕組み
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    with torch.no_grad():   # 推論だけなので勾配計算は不要
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,                      # 毎回同じ答えになる決定的モード
            pad_token_id=tokenizer.eos_token_id,
        )
    # 入力プロンプト部分を取り除き、生成された回答だけをデコード
    answer = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    return answer.strip()

# ⚠️ ここを、あなたのFAQデータに実際に含まれる質問に書き換えてください！
test_question = "営業時間を教えてください"

print(f"質問: {test_question}")
print(f"訓練前の回答 (Before): {ask_model(base_model, test_question)}")

# ==============================================================================
# ✅ ステップ3：あなたのFAQデータを訓練用に変換する
# ==============================================================================

lab2で作った `my_faq.txt`（Q:/A:形式）を再利用します。

lab2では生テキストをそのまま食べさせましたが、Instructモデルには
**「チャットテンプレート」** という正式な会話形式に変換して与えます。
（新入社員に、社内研修資料を正式なフォーマットで渡すイメージです）


In [ ]:
import os, re

# my_faq.txt を読み込み（無い場合はラーメン屋サンプルFAQで動きます）
if os.path.exists("my_faq.txt"):
    with open("my_faq.txt", "r", encoding="utf-8") as f:
        faq_raw = f.read()
    print(f"✅ my_faq.txt を読み込みました")
else:
    print("⚠️ my_faq.txt が見つかりません。サンプルFAQを使います。")
    faq_raw = """
Q: 営業時間を教えてください
A: 営業時間は11時から21時までです。スープがなくなり次第終了します。

Q: おすすめのメニューは何ですか？
A: 当店は塩ラーメン一本です。まずは塩ラーメンをお試しください。

Q: 大盛りはできますか？
A: 味のバランスを守るため、大盛りは承っておりません。

Q: 支払い方法は何が使えますか？
A: 食券制で、現金のみのお取り扱いです。

Q: 駐車場はありますか？
A: 店舗前に3台分の駐車場がございます。
"""

# 正規表現で Q:/A: のペアを抽出します
pairs = re.findall(r"Q[:：]\s*(.+?)\s*\nA[:：]\s*(.+?)(?=\n\s*\n|\n\s*Q[:：]|$)", faq_raw, re.S)
print(f"抽出されたQ&Aペア: {len(pairs)} 組")
print(f"例: Q={pairs[0][0][:30]} / A={pairs[0][1][:30]}")

# 各ペアを「チャットテンプレート形式」の訓練テキストに変換
system_msg = "あなたは親切なFAQアシスタントです。簡潔に日本語で答えてください。"
train_texts = []
for q, a in pairs:
    messages = [
        {"role": "system", "content": system_msg},
        {"role": "user", "content": q.strip()},
        {"role": "assistant", "content": a.strip()},
    ]
    train_texts.append(tokenizer.apply_chat_template(messages, tokenize=False))

print(f"\n--- 訓練テキストの例（モデルが実際に見る形式）---")
print(train_texts[0])

# ==============================================================================
# ✅ ステップ4：LoRAアダプタ（付箋）を取り付ける
# ==============================================================================

`peft` ライブラリで、モデルのアテンション層（Q,K,V,O — 今日ゼロから作ったあの部品です！）に
LoRAアダプタを取り付けます。元の5億パラメータは **凍結** され、付箋だけが訓練対象になります。


In [ ]:
lora_config = LoraConfig(
    r=8,                    # 付箋の「厚み」(ランク)。大きいほど表現力↑・メモリ↑
    lora_alpha=16,          # 付箋の影響の強さ（通常 r の2倍程度）
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],  # 付箋を貼る場所 = アテンション層！
    lora_dropout=0.05,      # 過学習防止
    task_type="CAUSAL_LM",  # タスク種別：因果言語モデル（=次トークン予測）
)

model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()
# → 「trainable」がLoRAの付箋。全体のわずか1%未満だけを訓練します！

# ==============================================================================
# ✅ ステップ5：ファインチューニング実行
# ==============================================================================

驚くことに、訓練ループは **今朝とまったく同じ4ステップ** です。

```text
 1. 予測と誤差計算 → 2. ゴミ掃除 → 3. 逆伝播 → 4. パラメータ更新
```

数万パラメータのBigramでも、5億パラメータのSLMでも、学習の原理は変わりません。
今日一日で学んだことが、そのまま実務スケールに通用するのです。


In [ ]:
# 訓練データをトークン化してテンソルにまとめます
# CPU環境では系列長を短めにして計算量を抑えます（FAQの1問答なら128トークンで十分）
max_length = 256 if device == 'cuda' else 128
enc = tokenizer(train_texts, padding=True, truncation=True, max_length=max_length, return_tensors="pt")
input_ids = enc["input_ids"].to(device)
attention_mask = enc["attention_mask"].to(device)

# 正解ラベル = 入力そのもの（次トークン予測のため）。
# ただしパディング（長さ合わせの詰め物）部分は -100 にして損失計算から除外します
labels = input_ids.clone()
labels[attention_mask == 0] = -100

optimizer = torch.optim.AdamW(model.parameters(), lr=2e-4)  # LoRAでは少し高めの学習率が定番

epochs = 20 if device == 'cuda' else 12   # CPUでは周回数を減らして時間短縮
batch_size = 4 if device == 'cuda' else 2
n = input_ids.shape[0]

model.train()
print(f"{len(train_texts)} 件のQ&Aで {epochs} エポックの訓練を開始します...")
print("（GPUなら1〜2分、CPUなら5〜15分程度が目安です。Lossが下がっていれば正常動作中）\n")

for epoch in range(epochs):
    perm = torch.randperm(n)          # 毎回シャッフル（偏り防止 — lab1で学んだ知恵）
    total_loss = 0.0
    for i in range(0, n, batch_size):
        idx = perm[i:i+batch_size]

        # 今朝と同じ4ステップ！
        out = model(input_ids=input_ids[idx],
                    attention_mask=attention_mask[idx],
                    labels=labels[idx])              # 1. 予測と誤差計算
        optimizer.zero_grad(set_to_none=True)        # 2. ゴミ掃除
        out.loss.backward()                          # 3. 逆伝播
        optimizer.step()                             # 4. 更新

        total_loss += out.loss.item()
    if epoch % 2 == 0 or epoch == epochs - 1:
        print(f"エポック {epoch:2d} | 平均Loss: {total_loss / ((n + batch_size - 1)//batch_size):.4f}")

model.eval()
print("\n🎉 ファインチューニング完了！")

# ==============================================================================
# ✅ ステップ6：訓練後のモデルに同じ質問をする (After)
# ==============================================================================
ステップ2の「Before」と比べてみましょう。あなたの会社のことを答えられるようになったでしょうか？


In [ ]:
print(f"質問: {test_question}\n")
print(f"訓練後の回答 (After): {ask_model(model, test_question)}")

# 他の質問も試してみましょう（あなたのFAQにある質問に書き換えてください）
for q in ["支払い方法は何が使えますか？", "駐車場はありますか？"]:
    print(f"\n質問: {q}")
    print(f"回答: {ask_model(model, q)}")

# ==============================================================================
# ✅ ステップ7：モデルの保存 — 持ち帰りの準備
# ==============================================================================

LoRAの付箋（アダプタ）だけを保存します。数十MB程度と軽量です。


In [ ]:
save_dir = "my_faq_slm_adapter"
model.save_pretrained(save_dir)      # LoRAアダプタを保存
tokenizer.save_pretrained(save_dir)  # トークナイザも一緒に保存

import subprocess
print(subprocess.run(["ls", "-lh", save_dir], capture_output=True, text=True).stdout)
print("✅ 保存完了！このフォルダをダウンロードすれば持ち帰れます")
print("   (JupyterLabの左サイドバーでフォルダを右クリック → Download)")

## 🏠 持ち帰り後の使い方 (Take-Home Guide)

### A. Pythonで再利用する（最も簡単）
自分のPCやサーバで、保存したアダプタを読み込むだけです。
どのベースモデルを使ったかはアダプタの設定ファイルに記録済みなので、自動で復元されます：

```python
from transformers import AutoTokenizer
from peft import AutoPeftModelForCausalLM

model = AutoPeftModelForCausalLM.from_pretrained("my_faq_slm_adapter")  # ベースモデル+付箋を自動復元
tokenizer = AutoTokenizer.from_pretrained("my_faq_slm_adapter")
```

### B. Ollamaでローカル実行する（応用）
[Ollama](https://ollama.com) を使うと、自分のPCでチャットアプリのように動かせます。
手順の概要（詳細は配布資料参照）：

1. アダプタを本体に統合: `merged = model.merge_and_unload()` して `save_pretrained("merged_model")`
2. [llama.cpp](https://github.com/ggml-org/llama.cpp) の `convert_hf_to_gguf.py` で **GGUF形式** に変換
3. `Modelfile` を書いて `ollama create my-faq-bot -f Modelfile`
4. `ollama run my-faq-bot` で起動！

### C. さらに良くするには
- **データを増やす**: Q&Aを100件、300件と増やすほど賢くなります
- **言い換えを加える**: 同じ質問の別の聞き方（「何時まで？」「営業時間は？」）を両方入れる
- **より大きなSLM**: GPU環境が用意できたら `model_name` を `Qwen/Qwen2.5-1.5B-Instruct` や
  `llm-jp/llm-jp-3-1.8b-instruct3` などに差し替えるだけで、このノートブックはそのまま動きます


<br>
<hr>

# ==============================================================================
# 🔌 ボーナス：MCP入門 — AIに「道具」を持たせる (Intro to MCP)
# ==============================================================================

ここまでのAIは「知っていることを話す」だけでした。しかし実務では、
**「最新の在庫を調べる」「予約システムに登録する」** など、AIに **行動** してほしい場面があります。

## MCP (Model Context Protocol) とは？

AIと外部ツール（データベース、API、社内システム）を繋ぐための **標準規格** です。
「AI用のUSB規格」と呼ばれます — 規格に従ってツールを作れば、
Claude・ChatGPTなどさまざまなAIから同じツールを使えます。

```text
 ユーザー「今日の営業時間は？」
      │
      ▼
 ┌─────────┐   「get_hours ツールを使おう」    ┌──────────────┐
 │   AI     │ ────────────────────────────▶ │ MCPサーバ      │
 │ (言語モデル)│ ◀──────────────────────────── │ (あなたのツール) │
 └─────────┘   「本日は11時〜21時です」       └──────────────┘
      │
      ▼
 ユーザーへ回答「本日の営業時間は11時から21時です」
```

**ポイント**: AIは文章を書くことしかできません（今日一日で実感した通り！）。
MCPは、AIが「ツールを使いたい」という **文章** を出力したら、それを実際の関数呼び出しに
変換してあげる仕組みです。

以下は先生によるデモです。ラーメン屋のMCPサーバを作ってみましょう。


In [ ]:
# MCPサーバを簡単に作れるライブラリ FastMCP をインストール
%pip install -q fastmcp

In [ ]:
# ラーメン屋MCPサーバの定義
# @mcp.tool を付けるだけで、関数がAIから使える「道具」になります
from fastmcp import FastMCP

mcp = FastMCP("ramen-shop")   # サーバに名前を付ける

@mcp.tool
def get_business_hours(day: str) -> str:
    """指定した曜日の営業時間を返します。dayは「月」〜「日」で指定。"""
    hours = {"月": "定休日", "火": "11:00-21:00", "水": "11:00-21:00",
             "木": "11:00-21:00", "金": "11:00-22:00", "土": "11:00-22:00", "日": "11:00-20:00"}
    return hours.get(day, "不明な曜日です")

@mcp.tool
def check_menu(item: str) -> str:
    """メニューにある商品か確認します。"""
    menu = {"塩ラーメン": 900, "ビール": 500}
    if item in menu:
        return f"{item}はございます。{menu[item]}円です。"
    return f"{item}は置いてません。うちは塩ラーメン一本です。"

print("🍜 ラーメン屋MCPサーバの定義完了！ ツール: get_business_hours, check_menu")

In [ ]:
# MCPクライアントからツールを呼び出してみます（AIの代わりに私たちが呼ぶテスト）
# ※ Jupyterでは await をそのまま書けます
from fastmcp import Client

async with Client(mcp) as client:
    # 1. AIはまず「どんな道具があるか」のリストを受け取ります
    tools = await client.list_tools()
    print("AIに見えるツール一覧:")
    for t in tools:
        print(f"  - {t.name}: {t.description}")

    # 2. AIが「金曜の営業時間を調べたい」と判断したと仮定して、ツールを実行
    result = await client.call_tool("get_business_hours", {"day": "金"})
    print(f"\nget_business_hours('金') → {result.content[0].text}")

    result = await client.call_tool("check_menu", {"item": "替え玉"})
    print(f"check_menu('替え玉')      → {result.content[0].text}")

## MCPサーバを実際にAIに接続するには？（持ち帰り資料）

上のデモでは私たちが手動でツールを呼びましたが、実運用では **AIが自動で** 呼び出します。

サーバをファイルとして保存し（下のセルで作成）、例えば **Claude Desktop** の設定に登録すると、
Claudeとの会話中に「金曜って何時まで営業？」と聞くだけで、Claudeが自動でこのツールを呼びます。

```json
// Claude Desktop の設定ファイル (claude_desktop_config.json) の例
{
  "mcpServers": {
    "ramen-shop": {
      "command": "python",
      "args": ["/path/to/ramen_mcp_server.py"]
    }
  }
}
```

社内システムのAPI・データベース検索・ファイル操作なども、同じパターンで
「AIの道具」にできます。**FAQはファインチューニング、リアルタイム情報はMCPツール** —
この使い分けが実務の定石です。


In [ ]:
# 持ち帰り用に、MCPサーバを独立したPythonファイルとして保存します
server_code = '''"""ラーメン屋MCPサーバ — ワークショップ持ち帰り版
実行方法: python ramen_mcp_server.py
"""
from fastmcp import FastMCP

mcp = FastMCP("ramen-shop")

@mcp.tool
def get_business_hours(day: str) -> str:
    """指定した曜日の営業時間を返します。dayは「月」〜「日」で指定。"""
    hours = {"月": "定休日", "火": "11:00-21:00", "水": "11:00-21:00",
             "木": "11:00-21:00", "金": "11:00-22:00", "土": "11:00-22:00", "日": "11:00-20:00"}
    return hours.get(day, "不明な曜日です")

@mcp.tool
def check_menu(item: str) -> str:
    """メニューにある商品か確認します。"""
    menu = {"塩ラーメン": 900, "ビール": 500}
    if item in menu:
        return f"{item}はございます。{menu[item]}円です。"
    return f"{item}は置いてません。うちは塩ラーメン一本です。"

if __name__ == "__main__":
    mcp.run()   # 標準入出力(stdio)でAIクライアントからの接続を待つ
'''

with open("ramen_mcp_server.py", "w", encoding="utf-8") as f:
    f.write(server_code)
print("✅ ramen_mcp_server.py を保存しました（ダウンロードして持ち帰れます）")

<br>
<hr>

# 🎓 ワークショップ完了：卒業おめでとうございます！

長い一日、お疲れ様でした！本日の到達点を振り返りましょう。

| | 午前：Bigram | 午後：Transformer | 最終章：LoRA SLM |
| :--- | :--- | :--- | :--- |
| **パラメータ数** | 約1万 | 約20万 | **1.5億〜5億** |
| **知識** | ラーメン屋の文字パターン | あなたのFAQの文字パターン | **日本語 + あなたのFAQ** |
| **作り方** | ゼロから | ゼロから | 事前学習済み + 微調整 |
| **実用性** | 教材 | 教材 | **持ち帰って使える！** |

### ✅ 今日学んだ最重要ポイント
1. **言語モデルの正体**: どんなに巨大でも「次のトークンの確率計算機」
2. **学習の4ステップ**: 予測 → 誤差 → 逆伝播 → 更新。Bigramでも5億パラメータでも同じ
3. **Transformerの心臓**: 自己アテンション（QKVの図書館）が文脈理解を生む
4. **実務の定石**: ゼロから作らず、事前学習済みSLMを **LoRAで微調整** する
5. **AIに行動させる**: 知識はファインチューニング、リアルタイム情報・操作は **MCPツール**

### 🏠 宿題 (Next Steps)
1. **データ拡充**: FAQを100件に増やして再訓練してみる
2. **モデル変更**: `model_name` を `Qwen/Qwen2.5-1.5B-Instruct` に変えてみる
3. **MCP実践**: `ramen_mcp_server.py` を自社の情報に書き換えて、Claude Desktopに繋いでみる

本日はありがとうございました！🍜
